# Lab 1: Build a searchable company handbook

## Business mission

The proposed solution is a retrieval-augmented generation system, or RAG. It can answer common HR questions in seconds instead of sending every question to an HR mailbox that may take up to 24 hours to respond. RAG is useful here because it retrieves current approved policy evidence, supports source citations, can be refreshed when policies change, and can refuse when evidence is missing. Your job is to turn the handbook PDF into searchable records that make fast, consistent, traceable answers possible.

## Why the business would build this

The business value is not the vector database or the language model by themselves. The value is a faster and more scalable HR service:

- Employees receive a useful first response in seconds instead of waiting in an email queue.
- HR specialists spend less time answering repeated policy questions and more time on complex cases.
- Answers are consistent because they use one approved policy collection.
- Citations let employees and HR verify where an answer came from.
- Policy changes can be indexed without retraining the language model.
- Unanswered questions become measurable demand that HR can use to improve policies and service.
- The organization can start with information retrieval and later connect approved workflows, subject to authorization and human review.

## What this lab builds

```text
company_handbook.pdf
  -> PDF loader
  -> pages with metadata
  -> recursive chunks
  -> embeddings
  -> Chroma vector store
  -> retrieved evidence
```

Run one cell at a time. Every code cell includes comments explaining the moving part and the production idea behind it.

## Before you run the API cells: configure `.env`

This lab uses two hosted providers. OpenAI creates the embeddings used for the handbook chunks and employee questions. Anthropic Claude writes the grounded HR answer from the retrieved evidence.

Create a `.env` file in the `week03` folder by copying `.env.example`, then add:

```text
OPENAI_API_KEY=your-openai-key
ANTHROPIC_API_KEY=your-anthropic-key
```

The notebook uses `load_dotenv()` to read these values without putting secrets in the code. Never paste a real key into a notebook, commit it to Git, or share it in a screenshot. In a production system, secrets are stored in a secret manager such as AWS Secrets Manager, Azure Key Vault, or Google Secret Manager.

## Exercise 1: Load the handbook PDF

**Mission:** Use a document loader to extract pages from a real PDF.

LangChain provides the loader. We are not manually searching for heading characters. The loader returns one document object per page and keeps page metadata for later citations.

In [6]:
# Import the PDF loader supplied by LangChain.
# PyPDFLoader extracts selectable text and keeps page information.
from langchain_community.document_loaders import PyPDFLoader

# Keep the source path in one variable so the ingestion job can change it later.
PDF_PATH = "sample_documents/company_handbook.pdf"

# Create the loader, then read the PDF pages into memory.
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print("Pages loaded:", len(pages))
print("First page metadata:", pages[0].metadata)
# page_content is the text extracted from the first page.
# [:500] previews only the first 500 characters so we can inspect the result.
print(pages[0].page_content[:500])


C:\Users\kserg\AppData\Local\Temp\ipykernel_30092\2916587490.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
incorrect startxref pointer(1)
parsing for Object Streams


Pages loaded: 21
First page metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-11T15:09:36-05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-11T15:09:36-05:00', 'subject': '(unspecified)', 'title': 'Utrains Company Handbook 2026', 'trapped': '/False', 'source': 'sample_documents/company_handbook.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1'}
Utrains Company Handbook | 2026.1 | Current
Page 1
 Utrains Company Handbook
Approved policy collection | Edition 2026.1 | Status: Current
Purpose and authority
This handbook describes workplace policies, responsibilities, procedures, exceptions, and escalation paths.
The current policy repository is authoritative when a policy or plan changes. Each section includes an owner,
version, status, and review context so an information system can return traceable evidence instead of an
unsupported gues


### PDF extraction checkpoint

Confirm that the output contains readable text and page metadata. A scanned image-only PDF may return little or no text. In production, that case goes through OCR or a managed document service such as Amazon Textract before chunking.

## Exercise 2: Inspect the extracted pages

**Mission:** Look at the page objects before splitting them.

A page is already a structured document record. The text is in `page_content`; the source and page number are in `metadata`. This is the common shape that downstream chunking can use.

In [7]:
# Inspect a few pages so we understand what the loader produced.
# We do not need to print the entire handbook.
for number, page in enumerate(pages[:3], start=1):
    print("\nPAGE", number)
    print("Metadata:", page.metadata)
    # page_content is the extracted text; [:300] previews its first 300 characters.
    print(page.page_content[:300])



PAGE 1
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-11T15:09:36-05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-11T15:09:36-05:00', 'subject': '(unspecified)', 'title': 'Utrains Company Handbook 2026', 'trapped': '/False', 'source': 'sample_documents/company_handbook.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1'}
Utrains Company Handbook | 2026.1 | Current
Page 1
 Utrains Company Handbook
Approved policy collection | Edition 2026.1 | Status: Current
Purpose and authority
This handbook describes workplace policies, responsibilities, procedures, exceptions, and escalation paths.
The current policy repository i

PAGE 2
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-11T15:09:36-05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-11T15:09:36-05:00', 'subject': '(unspecified)', 'title': 'Utrains Company Hand

## Exercise 3: Split pages into retrieval chunks

**Mission:** Create smaller pieces that can be embedded and retrieved.

Recursive splitting tries paragraph and sentence boundaries before making a smaller cut. The starting values are 500 characters and 50 characters of overlap so the behavior is easy to inspect. Production teams measure these settings and may use token-based limits instead.

In [8]:
# Import the splitter used by many LangChain RAG examples.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# These are a starting configuration, not universal truth.
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# The splitter tries larger boundaries first, such as paragraphs and sentences.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    # Try these boundaries in order, from the largest to the smallest.
    # A double line break means a blank line between paragraphs.
    # A single line break separates lines, and a period plus space ends a sentence.
    # A space separates words. The final empty separator allows a last-resort character split.
    separators=["\n\n", "\n", ". ", " ", ""],
)

# Split the page documents while retaining their source metadata.
chunks = splitter.split_documents(pages)

print("Chunks created:", len(chunks))


Chunks created: 62


In [9]:
# Print a few chunks so we can see what will become searchable rows.
# Notice that page metadata remains attached to each chunk.
for number, chunk in enumerate(chunks[:5]):
    print("\nCHUNK", number)
    print("Metadata:", chunk.metadata)
    # page_content is the chunk text; [:300] previews it without printing everything.
    print(chunk.page_content)



CHUNK 0
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-11T15:09:36-05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-11T15:09:36-05:00', 'subject': '(unspecified)', 'title': 'Utrains Company Handbook 2026', 'trapped': '/False', 'source': 'sample_documents/company_handbook.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1'}
Utrains Company Handbook | 2026.1 | Current
Page 1
 Utrains Company Handbook
Approved policy collection | Edition 2026.1 | Status: Current
Purpose and authority
This handbook describes workplace policies, responsibilities, procedures, exceptions, and escalation paths.
The current policy repository is authoritative when a policy or plan changes. Each section includes an owner,
version, status, and review context so an information system can return traceable evidence instead of an

CHUNK 1
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)',

### Chunking checkpoint

Read the printed chunks as a first manual quality check. Ask four questions:

1. Does each chunk contain a complete idea, or was a rule separated from its exception?
2. Is the chunk close to the target size without being cut in the middle of a word or sentence?
3. Does the next chunk repeat enough context where a split occurred?
4. Did the source and page metadata remain attached?

If a chunk fails one of these checks, chunking may be broken for this document. Change the splitter settings, inspect more examples, and evaluate the change against real questions. Printing a few chunks is a useful first check, but production teams also measure retrieval quality with a repeatable test set.

## Exercise 4: Create embeddings and a local vector store

**Mission:** Convert the chunks into vectors and store them in Chroma.

Chroma is a local vector database that makes the retrieval mechanics visible. In production, the same index contract could use OpenSearch, pgvector, Pinecone, Qdrant, or a managed cloud search service.

In [10]:
# OpenAIEmbeddings sends each chunk to the embedding model.
# The same model must embed both documents and future questions.
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
load_dotenv()

EMBEDDING_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Chroma stores the chunk text, its metadata, and the vector representation.
# documents=chunks means each chunk is one searchable record.
# embedding=embeddings tells Chroma how to turn text into vectors.
# collection_name gives this group of records a name inside Chroma.
# persist_directory tells Chroma to save the collection on disk.
# Page and source metadata travel with each chunk for citations and filters.
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="company_handbook",
    persist_directory="chroma_data/handbookdb",
)

print("Indexed chunks:", len(chunks))


Indexed chunks: 62


### Chroma storage checkpoint

The storage step is `Chroma.from_documents(...)`. It loops through the chunks, uses the embedding model to create one vector per chunk, and writes three related pieces of information into the `company_handbook` collection: the original chunk text, its vector, and its metadata such as the source and page. The returned `vector_store` object is our handle for searching those stored records.

The notebook does not call a separate `save` function. `from_documents` creates the collection and inserts the records for us. `persist_directory="chroma_data/lab1"` tells Chroma to write its local database files under the project folder, so the collection can be reopened later. Without that option, the collection is held in memory and disappears when the Python process ends. In a production vector database, the same operation would write to a managed index or collection.

In [11]:
# Load the keys from week03/.env before any hosted API call.
import os
from dotenv import load_dotenv

load_dotenv()

# Check that the keys exist without printing their secret values.
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing. Add it to week03/.env.")

if not os.getenv("ANTHROPIC_API_KEY"):
    raise ValueError("ANTHROPIC_API_KEY is missing. Add it to week03/.env.")

print("API keys loaded from .env")

API keys loaded from .env


### Inspect one stored record

This cell asks Chroma for exactly one stored record and prints the complete response. That lets us see all returned fields, including the document text, metadata, record ID, and full embedding vector. The vector is long because it contains one number for every embedding dimension.

In [12]:
# Get one record from the Chroma collection.
stored_record = vector_store.get(
    limit=1,
    include=["documents", "metadatas", "embeddings"],
)

# Print the complete one-row response from Chroma.
# This includes documents, metadatas, embeddings, and IDs.
print(stored_record)

{'ids': ['acf3fd8f-e0dc-466f-b258-48390a740c4f'], 'embeddings': array([[-0.02275085,  0.02320862,  0.03056335, ..., -0.00281525,
         0.01849365, -0.01994324]], shape=(1, 1536)), 'documents': ['Utrains Company Handbook | 2026.1 | Current\nPage 1\n Utrains Company Handbook\nApproved policy collection | Edition 2026.1 | Status: Current\nPurpose and authority\nThis handbook describes workplace policies, responsibilities, procedures, exceptions, and escalation paths.\nThe current policy repository is authoritative when a policy or plan changes. Each section includes an owner,\nversion, status, and review context so an information system can return traceable evidence instead of an'], 'uris': None, 'included': ['documents', 'metadatas', 'embeddings'], 'data': None, 'metadatas': [{'subject': '(unspecified)', 'page_label': '1', 'creationdate': '2026-09-11T15:09:36-05:00', 'total_pages': 21, 'source': 'sample_documents/company_handbook.pdf', 'trapped': '/False', 'page': 0, 'creator': '(unsp

## Exercise 5: Retrieve evidence for a business question

**Mission:** Search the handbook before asking a language model to answer.

The model should receive retrieved evidence, not the entire PDF. We inspect the results first so we can verify the source pages and understand what the model will see.

### How the question becomes searchable

Chroma does not choose an embedding model by itself. In Exercise 4 we passed `embeddings` into `Chroma.from_documents`. That object uses OpenAI `text-embedding-3-small`. When we call `similarity_search(QUESTION, k=TOP_K)`, LangChain sends the question to that same embedding object, receives one vector for the question, and gives the vector to Chroma. Chroma compares it with the stored chunk vectors and returns the closest matches. The document chunks and questions must use the same embedding model so their vectors live in the same vector space.

### Retrieval methods to know

LangChain provides the Python method, and Chroma performs the search over the stored vectors. The most common starting point is similarity search:

- `similarity_search(question, k=3)` is the normal application choice. LangChain embeds the question, Chroma compares that vector with the stored chunk vectors, and the three closest chunks are returned.
- `similarity_search_with_score(question, k=3)` does the same search but also returns a number for each result. We use this when inspecting retrieval quality or deciding whether a result is relevant enough.
- `similarity_search_by_vector(vector, k=3)` is a lower-level option. Your code has already turned the question into a vector, and you pass that vector directly. Most beginner applications should use the question-based method instead.
- `max_marginal_relevance_search(question, k=3, fetch_k=10)` first finds several close chunks and then tries to remove repeated or nearly identical chunks. It is useful when ordinary similarity search returns redundant passages, but it is not the default starting point.
- Hybrid retrieval combines vector similarity with keyword search or metadata filters. Production systems often add this when exact identifiers, policy codes, names, or dates are important.

### What hybrid search means

Vector search is good at meaning. It can match a question such as `Can I claim this travel cost?` with a policy that uses different words such as `eligible transportation expenses`. Keyword search is good at exact text. It can find a policy code such as `TRAVEL-204` or a benefit plan such as `HMO-204`. Hybrid search uses both signals, then combines or reranks the results before sending evidence to the model.

A production HR assistant may use vector search for the meaning of a question, keyword search for exact names and codes, and metadata filters for policy status, region, or department. The result is usually more reliable than relying on either search type alone.

For this lab, remember the order: use `similarity_search` for the normal request path, use `similarity_search_with_score` to inspect and evaluate it, and learn hybrid retrieval as the production option when exact terms and meaning both matter.

In [31]:
# This is the employee question our retrieval system must support.
#QUESTION = "How many vacation days do employees receive?"
QUESTION="How many weeks vaccation or pto do I have"
#QUESTION = "what is the parental leave policy?"
# TOP_K is the maximum number of chunks we request. It does not guarantee that many results.
TOP_K = 3

# Chroma embeds the question and returns the closest chunks.
results = vector_store.similarity_search(QUESTION, k=TOP_K)
print("Results returned:", len(results), "of up to", TOP_K)

for number, result in enumerate(results, start=1):
    print("\nRESULT", number)
    print("Metadata:", result.metadata)
    # page_content is the text stored in this retrieved Document.
    # [:500] is Python slicing: print only the first 500 characters
    # so a long chunk does not fill the screen. The full text remains available.
    print(result.page_content[:500])


Results returned: 3 of up to 3

RESULT 1
Metadata: {'subject': '(unspecified)', 'trapped': '/False', 'moddate': '2026-09-11T15:09:36-05:00', 'creator': '(unspecified)', 'page': 1, 'creationdate': '2026-09-11T15:09:36-05:00', 'source': 'sample_documents/company_handbook.pdf', 'title': 'Utrains Company Handbook 2026', 'page_label': '2', 'keywords': '', 'author': '(anonymous)', 'producer': 'ReportLab PDF Library - (opensource)', 'total_pages': 21}
Utrains Company Handbook | 2026.1 | Current
Page 2
1. Time Off and Annual Leave
Scope and policy
Full-time employees receive 20 days of annual leave each calendar year. Requests should normally be
submitted through the HR portal at least five business days before the first day away. Managers should
respond within two business days.
Procedure and responsibilities
The employee checks the available balance, submits the requested dates, and waits for manager approval

RESULT 2
Metadata: {'trapped': '/False', 'source': 'sample_documents/company_handb

## Optional retrieval experiment: simple hybrid search

**Mission:** Combine meaning-based search with an exact keyword match.

This small example shows the idea behind hybrid search. The vector search finds text with a similar meaning. The keyword check finds an exact policy code. Production search services can combine these signals more efficiently and rank the combined results.

In [32]:
# Start with semantic vector search.
hybrid_question = "What is the specialist copay for HMO-204?"
semantic_results = vector_store.similarity_search(hybrid_question, k=3)

# Also look for the exact policy code in the stored chunk text.
keyword = "HMO-204"
keyword_results = []
for chunk in chunks:
    if keyword.lower() in chunk.page_content.lower():
        keyword_results.append(chunk)

# Add semantic results first, then add keyword results that are not duplicates.
hybrid_results = []
for result in semantic_results:
    hybrid_results.append(result)
for result in keyword_results:
    already_added = False
    for existing in hybrid_results:
        if existing.page_content == result.page_content:
            already_added = True
    if not already_added:
        hybrid_results.append(result)

for result in hybrid_results:
    print(result.metadata)
    print(result.page_content)
    print()

{'keywords': '', 'trapped': '/False', 'total_pages': 21, 'producer': 'ReportLab PDF Library - (opensource)', 'page': 5, 'moddate': '2026-09-11T15:09:36-05:00', 'page_label': '6', 'subject': '(unspecified)', 'creator': '(unspecified)', 'author': '(anonymous)', 'source': 'sample_documents/company_handbook.pdf', 'creationdate': '2026-09-11T15:09:36-05:00', 'title': 'Utrains Company Handbook 2026'}
Utrains Company Handbook | 2026.1 | Current
Page 6
5. Health Benefits
Scope and policy
Employees may choose individual or family coverage during the enrollment window. For the 2026 plan year,
HMO-204 has a $35 specialist copay and PPO-440 has a $60 specialist copay. Current premiums,
deductibles, and network details are published in the benefits portal.
Procedure and responsibilities
The benefits portal controls current prices because plans can change during the year. An employee should

{'total_pages': 21, 'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'page': 5

## Exercise 6: Ask the language model using retrieved context

**Mission:** Give the model only the evidence found by the retriever and ask for a grounded answer.

The answer is only as reliable as the retrieved context. If the context does not contain the answer, the application should say that evidence is insufficient.

In [33]:
# Build the context one result at a time so we can see what is being sent to the model.

context = ""
for result in results:
    context = context + result.page_content + "\n\n"

# Keep source metadata available for citations in a real application.
sources = []
for result in results:
    sources.append(result.metadata)

# This prompt makes the grounding rule explicit.
prompt = f"""
Answer the question using only the context below.
If the context does not contain the answer, say that there is not enough evidence.
Give direct answer to users.

Context:
{context}

Question:
{QUESTION}
"""

from langchain_anthropic import ChatAnthropic
model = ChatAnthropic(model="claude-haiku-4-5")
answer = model.invoke(prompt)

print(answer.content)
#print("Sources:", sources)


Based on the context provided:

**Full-time employees receive 20 days of annual leave each calendar year.**

This is the standard vacation/PTO entitlement mentioned in the handbook. 

Note: This is separate from parental leave, which offers 20 weeks for eligible employees (after 6 months of continuous service) in cases of birth, adoption, or child placement.

If you need clarification on your specific situation or have questions about other types of leave, you may need to contact your HR department.


## Lab 1 checkpoint

You have now followed the production-shaped request path: a PDF loader produced page documents, a splitter created traceable chunks, an embedding model created vectors, Chroma stored the index, retrieval selected evidence, and an LLM answered from that evidence.

The local choices are replaceable. The pipeline contract is the important part.

## Production handoff: complex documents

This lab uses a text-based PDF so the loading and retrieval steps remain visible. A real corpus may also contain scanned PDFs, tables, images, audio, video, DOCX, HTML, and XML.

For those sources, the ingestion pipeline routes each format to the appropriate managed capability before chunking:

```text
scanned PDF or image -> OCR and layout extraction
PDF with tables       -> document analysis
DOCX or HTML          -> format-aware parser
audio or video        -> transcription and time-based chunks
                      -> normalized document records
                      -> chunking, embeddings, and indexing
```

On AWS, services such as Amazon Textract and Bedrock Data Automation can handle OCR, layout, tables, and multimodal extraction. The production team still validates the output, preserves page and source metadata, and quarantines failed documents. The local PDF loader is the learning implementation of the same ingestion stage.